# ANSD ablation — noise-aug vs distillation

Lead: distill components tiny (L_logit≈0). Is gain from NOISE-as-aug (CE on noisy view)
or two-level distill? vs full ANSD 76.50 & baseline 73.6. Set E (60=cheap ranking).


In [ ]:
MODEL='ablation_ansd'; E=60


In [ ]:
REPO='https://github.com/almaas-izdihar/code-ansd'; DIR='/content/code-ansd'; DATA='/content/data'
import os, subprocess
if not os.path.exists(DIR): subprocess.run(f'git clone {REPO} {DIR}', shell=True, check=True)
subprocess.run(f'git -C {DIR} pull origin main', shell=True, check=True); os.chdir(DIR)
from utils.colab import gh_token, download_cifar100, run_training, push_results
GH_TOKEN = gh_token()


In [ ]:
download_cifar100(DATA)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## baseline — plain CE clean (ref ~73.6)


In [ ]:
cmd=(f'python3 main.py  --data_type cifar100 --data_path {DATA} --classifier_type ResNet18 '
     f'--batch_size 128 --end_epoch {E} --workers 2 --seed 2024 --gpu 0 '
     f'--experiments_dir models --experiment_type abl_baseline')
run_training(cmd)


## noiseCE-only — alpha=beta=0 → isolate NOISE-as-augmentation (KEY)


In [ ]:
cmd=(f'python3 main.py --ANSD --alpha 0 --beta 0 --lambda_noise 1.0 --T 1.0 --ce_view noise --data_type cifar100 --data_path {DATA} --classifier_type ResNet18 '
     f'--batch_size 128 --end_epoch {E} --workers 2 --seed 2024 --gpu 0 '
     f'--experiments_dir models --experiment_type abl_noiseCE')
run_training(cmd)


## logit-only — alpha=1 beta=0


In [ ]:
cmd=(f'python3 main.py --ANSD --alpha 1 --beta 0 --lambda_noise 1.0 --T 1.0 --ce_view noise --data_type cifar100 --data_path {DATA} --classifier_type ResNet18 '
     f'--batch_size 128 --end_epoch {E} --workers 2 --seed 2024 --gpu 0 '
     f'--experiments_dir models --experiment_type abl_logit')
run_training(cmd)


## feat-only — alpha=0 beta=1


In [ ]:
cmd=(f'python3 main.py --ANSD --alpha 0 --beta 1 --lambda_noise 1.0 --T 1.0 --ce_view noise --data_type cifar100 --data_path {DATA} --classifier_type ResNet18 '
     f'--batch_size 128 --end_epoch {E} --workers 2 --seed 2024 --gpu 0 '
     f'--experiments_dir models --experiment_type abl_feat')
run_training(cmd)


## push all ablation logs


In [ ]:
push_results('code-ansd', DIR, MODEL, 'abl_', GH_TOKEN)
